# Set up the RL-agent environment (DeepMind Lab)

Builds `lab/` (the `google-deepmind/lab` git submodule, pinned to the exact commit this
project was developed and tested against) into a `deepmind_lab` Python module importable
alongside PyTorch, for the paper's RL-agent phase (see README's "Roadmap: beyond the
supervised network").

`lab/` is a ~3GB Quake III-derived 3D engine (C++ + Lua) written for a 2018-era toolchain.
It does not build out of the box on a modern machine (GCC 14+, Bazel 9, Python 3.12+) —
this notebook applies the small set of fixes that make it build here, then packages and
installs the result as a normal pip wheel. Nothing here modifies `lab/`'s pinned submodule
commit in git; the patches are applied to the working tree only (`git submodule status`
will show it as locally modified after running this, which is expected).

Run this once before any RL code (vision module, grid network, policy LSTM, training loop)
will be able to `import deepmind_lab`.


## 1. Make sure the `lab/` submodule is checked out

In [1]:
import os
import sys
from pathlib import Path

# Match notebooks 00-05: resolve paths relative to the repo root regardless of
# whether this runs interactively (cwd == notebooks/) or headlessly via nbconvert
# (cwd also defaults to notebooks/), so "lab/..." always lands in the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import subprocess

subprocess.run(["git", "submodule", "update", "--init", "lab"], check=True)


CompletedProcess(args=['git', 'submodule', 'update', '--init', 'lab'], returncode=0)

## 2. Apply the toolchain patches

Six small files, applied directly here rather than kept as a separate patches directory —
none of this changes DeepMind Lab's own logic, it only makes the *build* work on a modern
toolchain.


### `lab/WORKSPACE`

- `com_google_absl` pinned to release `20230802.1` (was floating on `master.zip`, which now
  requires Bzlmod-only `rules_cc` that this WORKSPACE-based build doesn't have).
- Eigen swapped from a 2016 bitbucket dev snapshot to release `3.4.0` (the old one fails
  under GCC 15's stricter template-body checking, `-Wtemplate-body`).
- `glib_archive` swapped from a from-source autotools build to the system's `libglib2.0-dev`
  (`apt install libglib2.0-dev`) — the vendored glib-2.55.1's own `./configure` can't run its
  compiled test binaries in a sandboxed/containerized build.
- `libxml_archive` gets a `patch_cmds` stripping a stray `#define LIBXML_LZMA_ENABLED` that
  survives `--without-lzma` in this release (a mismatch between it and `config.h`'s correctly
  unset `HAVE_LIBLZMA`/`HAVE_LZMA_H`) — a genrule can't durably patch the persistent extracted
  source, only `http_archive`'s own `patch_cmds` can.


In [2]:
%%writefile lab/WORKSPACE
workspace(name = "org_deepmind_lab")

load("@bazel_tools//tools/build_defs/repo:http.bzl", "http_archive")
load("@//:python_system.bzl", "python_repo")

http_archive(
    name = "com_google_googletest",
    strip_prefix = "googletest-main",
    urls = ["https://github.com/google/googletest/archive/main.zip"],
)

http_archive(
    name = "bazel_skylib",
    strip_prefix = "bazel-skylib-main",
    urls = ["https://github.com/bazelbuild/bazel-skylib/archive/main.zip"],
)

http_archive(
    name = "com_google_absl",
    strip_prefix = "abseil-cpp-20230802.1",
    urls = ["https://github.com/abseil/abseil-cpp/archive/refs/tags/20230802.1.zip"],
)

http_archive(
    name = "com_google_absl_py",
    strip_prefix = "abseil-py-main",
    urls = ["https://github.com/abseil/abseil-py/archive/main.zip"],
)

http_archive(
    name = "enum34_archive",
    build_file = "@com_google_absl_py//third_party:enum34.BUILD",
    sha256 = "8ad8c4783bf61ded74527bffb48ed9b54166685e4230386a9ed9b1279e2df5b1",
    urls = [
        "https://mirror.bazel.build/pypi.python.org/packages/bf/3e/31d502c25302814a7c2f1d3959d2a3b3f78e509002ba91aea64993936876/enum34-1.1.6.tar.gz",
        "https://pypi.python.org/packages/bf/3e/31d502c25302814a7c2f1d3959d2a3b3f78e509002ba91aea64993936876/enum34-1.1.6.tar.gz",
    ],
)

http_archive(
    name = "funcsigs_archive",
    build_file = "@//bazel:funcsigs.BUILD",
    strip_prefix = "funcsigs-1.0.2",
    urls = [
        "https://pypi.python.org/packages/94/4a/db842e7a0545de1cdb0439bb80e6e42dfe82aaeaadd4072f2263a4fbed23/funcsigs-1.0.2.tar.gz",
    ],
)

http_archive(
    name = "eigen_archive",
    build_file = "@//bazel:eigen.BUILD",
    strip_prefix = "eigen-3.4.0",
    urls = [
        "https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.tar.gz",
    ],
)

# Uses the system's glib development package (installed via apt) rather than
# vendoring & autoconf-building glib-2.55.1 from source, which fails to
# configure in this environment ("cannot run C compiled programs").
new_local_repository(
    name = "glib_archive",
    build_file = "@//bazel:glib.BUILD",
    path = "/usr",
)

http_archive(
    name = "jpeg_archive",
    build_file = "@//bazel:jpeg.BUILD",
    sha256 = "2303a6acfb6cc533e0e86e8a9d29f7e6079e118b9de3f96e07a71a11c082fa6a",
    strip_prefix = "jpeg-9d",
    urls = ["http://www.ijg.org/files/jpegsrc.v9d.tar.gz"],
)

http_archive(
    name = "libxml_archive",
    build_file = "@//bazel:libxml.BUILD",
    sha256 = "f63c5e7d30362ed28b38bfa1ac6313f9a80230720b7fb6c80575eeab3ff5900c",
    strip_prefix = "libxml2-2.9.7",
    urls = [
        "https://mirror.bazel.build/xmlsoft.org/sources/libxml2-2.9.7.tar.gz",
        "http://xmlsoft.org/sources/libxml2-2.9.7.tar.gz",
    ],
    # The shipped include/libxml/xmlversion.h has LIBXML_LZMA_ENABLED
    # #defined regardless of ./configure --without-lzma (a mismatch between
    # it and config.h's correctly-unset HAVE_LIBLZMA/HAVE_LZMA_H). The
    # BUILD's own genrule regenerates a fixed copy, but the compiler's
    # include search still finds this original first, so patch it here at
    # fetch time -- a genrule can't durably mutate the persistent extracted
    # source, only produce new declared outputs.
    patch_cmds = [
        "sed -i -e '/#define LIBXML_LZMA_ENABLED/d' include/libxml/xmlversion.h",
    ],
)

http_archive(
    name = "png_archive",
    build_file = "@//bazel:png.BUILD",
    sha256 = "c2c50c13a727af73ecd3fc0167d78592cf5e0bca9611058ca414b6493339c784",
    strip_prefix = "libpng-1.6.37",
    urls = [
        "https://mirror.bazel.build/github.com/glennrp/libpng/archive/v1.6.37.zip",
        "https://github.com/glennrp/libpng/archive/v1.6.37.zip",
    ],
)

http_archive(
    name = "zlib_archive",
    build_file = "@//bazel:zlib.BUILD",
    sha256 = "c3e5e9fdd5004dcb542feda5ee4f0ff0744628baf8ed2dd5d66f8ca1197cb1a1",
    strip_prefix = "zlib-1.2.11",
    urls = [
        "https://mirror.bazel.build/zlib.net/zlib-1.2.11.tar.gz",
        "https://zlib.net/zlib-1.2.11.tar.gz",
    ],
)

http_archive(
    name = "six_archive",
    build_file = "@//bazel:six.BUILD",
    sha256 = "30639c035cdb23534cd4aa2dd52c3bf48f06e5f4a941509c8bafd8ce11080259",
    strip_prefix = "six-1.15.0",
    urls = [
        "https://mirror.bazel.build/pypi.python.org/packages/source/s/six/six-1.15.0.tar.gz",
        "https://pypi.python.org/packages/source/s/six/six-1.15.0.tar.gz",
    ],
)

http_archive(
    name = "lua_archive",
    build_file = "@//bazel:lua.BUILD",
    sha256 = "2640fc56a795f29d28ef15e13c34a47e223960b0240e8cb0a82d9b0738695333",
    strip_prefix = "lua-5.1.5/src",
    urls = [
        "https://mirror.bazel.build/www.lua.org/ftp/lua-5.1.5.tar.gz",
        "https://www.lua.org/ftp/lua-5.1.5.tar.gz",
    ],
)

http_archive(
    name = "dm_env_archive",
    build_file = "@//bazel:dm_env.BUILD",
    strip_prefix = "dm_env-master",
    urls = ["https://github.com/deepmind/dm_env/archive/master.zip"],
)

http_archive(
    name = "tree_archive",
    build_file = "@//bazel:tree.BUILD",
    strip_prefix = "tree-master",
    urls = ["https://github.com/deepmind/tree/archive/master.zip"],
)

http_archive(
    name = "pybind11_archive",
    build_file = "@//bazel:pybind11.BUILD",
    strip_prefix = "pybind11-master",
    urls = ["https://github.com/pybind/pybind11/archive/master.zip"],
)

# TODO: Replace with hermetic build
new_local_repository(
    name = "sdl_system",
    build_file = "@//bazel:sdl.BUILD",
    path = "/usr",
)

python_repo(
    name = "python_system",
    py_version = "PY3",
)


Overwriting lab/WORKSPACE


### `lab/bazel/glib.BUILD`

Rewritten for the system-glib swap above (mirrors the existing `bazel/sdl.BUILD` pattern
already used for `sdl_system`).


In [3]:
%%writefile lab/bazel/glib.BUILD
# Description:
#   Build rule for GLib, using the system-installed development package
#   instead of vendoring & autoconf-building an old glib source snapshot
#   (which fails to configure in this environment).
#   Compiler and linker flags found with `pkg-config --cflags --libs glib-2.0`.

cc_library(
    name = "glib",
    hdrs = glob([
        "include/glib-2.0/**/*.h",
        "lib/x86_64-linux-gnu/glib-2.0/include/*.h",
    ]),
    includes = [
        "include/glib-2.0",
        "lib/x86_64-linux-gnu/glib-2.0/include",
    ],
    linkopts = ["-lglib-2.0"],
    visibility = ["//visibility:public"],
)


Overwriting lab/bazel/glib.BUILD


### `lab/bazel/libxml.BUILD`

Drops `xzlib.c` from `srcs` (liblzma isn't available; `configure` is already passed
`--without-lzma` above, matching upstream's own `Makefile.am`, which only builds `xzlib.c`
`if WITH_LZMA` — the hand-written BUILD file here didn't replicate that conditional).


In [4]:
%%writefile lab/bazel/libxml.BUILD
genrule(
    name = "gen_configure",
    srcs = [
        "Makefile.in",
        "config.guess",
        "config.h.in",
        "config.sub",
        "configure",
        "entities.c",
        "install-sh",
        "libxml-2.0.pc.in",
        "libxml-2.0-uninstalled.pc.in",
        "libxml2-config.cmake.in",
        "libxml.spec.in",
        "ltmain.sh",
        "missing",
        "xml2-config.in",
        "doc/Makefile.in",
        "doc/devhelp/Makefile.in",
        "doc/examples/Makefile.in",
        "example/Makefile.in",
        "include/Makefile.in",
        "include/libxml/Makefile.in",
        "include/libxml/xmlversion.h.in",
        "python/Makefile.in",
        "python/setup.py.in",
        "python/tests/Makefile.in",
        "xstc/Makefile.in",
    ],
    outs = [
        "config.h",
        "include/libxml/xmlversion.h",
    ],
    cmd = "./$(location configure) --silent --without-lzma " +
          "&& cp --verbose -- config.h $(location config.h) " +
          "&& cp --verbose -- include/libxml/xmlversion.h $(location include/libxml/xmlversion.h)",
)

cc_library(
    name = "libxml",
    srcs = [
        "HTMLparser.c",
        "HTMLtree.c",
        "SAX.c",
        "SAX2.c",
        "buf.c",
        "c14n.c",
        "catalog.c",
        "chvalid.c",
        "debugXML.c",
        "dict.c",
        "encoding.c",
        "entities.c",
        "error.c",
        "globals.c",
        "hash.c",
        "legacy.c",
        "list.c",
        "nanoftp.c",
        "nanohttp.c",
        "parser.c",
        "parserInternals.c",
        "pattern.c",
        "relaxng.c",
        "schematron.c",
        "threads.c",
        "tree.c",
        "uri.c",
        "valid.c",
        "xinclude.c",
        "xlink.c",
        "xmlIO.c",
        "xmlmemory.c",
        "xmlmodule.c",
        "xmlreader.c",
        "xmlregexp.c",
        "xmlsave.c",
        "xmlschemas.c",
        "xmlschemastypes.c",
        "xmlstring.c",
        "xmlunicode.c",
        "xmlwriter.c",
        "xpath.c",
        "xpointer.c",
        # xzlib.c requires liblzma, which isn't available; configure is
        # already passed --without-lzma above, matching upstream's own
        # Makefile.am (xzlib.c is only built `if WITH_LZMA`).
    ],
    hdrs = [
        "config.h",
        "include/libxml/xmlversion.h",
    ] + glob(
        [
            "*.h",
            "include/libxml/*.h",
        ],
        exclude = [
            # Exclude the pre-made version that ships with the tarball,
            # use the genrule output from ./configure instead.
            "include/libxml/xmlversion.h",
        ],
    ),
    copts = [
        "-D_REENTRANT",
        "-DHAVE_CONFIG_H",
        "-w",
    ],
    includes = [
        ".",
        "include",
    ],
    linkopts = ["-pthread"],
    textual_hdrs = ["trionan.c"],
    visibility = ["//visibility:public"],
)


Overwriting lab/bazel/libxml.BUILD


### `lab/python/pip_package/__init__.py`

Three fixes for the pip-installable module to work under Python 3.14 in a plain `env1`
interpreter (not a `bazel run` wrapper):

1. `imp.load_dynamic` was removed in Python 3.12 — replaced with `importlib.util`.
2. The extension module must be registered in `sys.modules['deepmind_lab']` itself (not kept
   as a private attribute of this wrapper package): `dmlab_module.c`'s `Lab.__init__` looks
   itself up via `PyImport_AddModule("deepmind_lab")` at call time to recover its own
   runfiles-path state. If `sys.modules['deepmind_lab']` points at this wrapper instead of the
   loaded extension, that self-lookup finds the wrong object and `Lab()` fails with
   `"Require runfiles_directory!"` even after calling `set_runfiles_path`.
3. `set_runfiles_path` is called explicitly, pointed at this package's own installed
   directory (where `baselab/` — the bundled assets/maps/scripts — lives) — the module's
   built-in `__file__`-based auto-detection doesn't reliably fire when loaded this way instead
   of via a normal `import`.


In [5]:
%%writefile lab/python/pip_package/__init__.py
"""The DeepMind Lab native module and dm_env bindings.

The native module is loaded from the DSO deepmind_lab.so. It provides
complete access to the functionality of DeepMind Lab. The API of this
module and the "Lab" type is documented in docs/users/python_api.md.
Use it as follows:

  import deepmind_lab

  lab = deepmind_lab.Lab(level='lt_chasm', observations=['RGB'])
  # ...

The "dm_env bindings" module provides bindings to the "dm_env" API. The module
is exposed as "deepmind_lab.dmenv_module". It requires the "dm_env" module; the
PIP "dmenv_module" extra describes that dependency. Only a subset of features
is accessible this way. Use the module as follows:

  from deepmind_lab import dmenv_module as dm_env_lab

  lab = dm_env_lab.Lab(level='lt_chasm', observation_names=['RGB'], config={})
  # ...
"""

import os
import sys
import importlib.util
from deepmind_lab import dmenv_module
import pkg_resources

# imp.load_dynamic (removed in Python 3.12) used to both load the compiled
# extension AND install it into sys.modules[name]. That second part matters
# here: dmlab_module.c's Lab.__init__ looks itself up at call time via
# PyImport_AddModule("deepmind_lab") to recover its own per-module state
# (in particular the runfiles path set below). If sys.modules['deepmind_lab']
# is left pointing at this wrapper package instead of the loaded extension,
# that self-lookup finds the wrong object and Lab() fails with
# "Require runfiles_directory!" even after calling set_runfiles_path. So the
# extension module must be installed under the 'deepmind_lab' name itself,
# not kept as a private attribute of the wrapper package.
_so_path = pkg_resources.resource_filename(__name__, 'deepmind_lab.so')
_spec = importlib.util.spec_from_file_location('deepmind_lab', _so_path)
_deepmind_lab = importlib.util.module_from_spec(_spec)
sys.modules['deepmind_lab'] = _deepmind_lab
_spec.loader.exec_module(_deepmind_lab)

# Re-attach the pieces this wrapper package normally provides, now as
# attributes of the extension module that has taken over the 'deepmind_lab'
# name in sys.modules.
_deepmind_lab.dmenv_module = dmenv_module
_deepmind_lab.set_runfiles_path(os.path.dirname(os.path.abspath(__file__)))


Overwriting lab/python/pip_package/__init__.py


### `lab/.bazelversion` and `lab/.bazelrc`

- `.bazelversion` pins Bazel to `6.5.0` via bazelisk — Bazel 9 (the current default) removed
  native `py_binary`/`py_library`, which this WORKSPACE-based build still relies on.
- `.bazelrc` bakes in the remaining fixes as flags rather than source changes:
  `--spawn_strategy=local` (sandboxing fails when running as root in a container, with no user
  namespaces), `-std=gnu11` for host and target (GCC 15 defaults to a C standard where
  `bspc`'s `typedef enum {false, true}` no longer compiles — `false`/`true` became C23
  keywords), and `-Wno-error=` for three diagnostics GCC 14+ promoted to hard errors by
  default (`incompatible-pointer-types`, `implicit-function-declaration`, `int-conversion`) —
  the vendored 1990s/2000s C (`q3map2/picomodel`, etc.) predates that and is harmless here.


In [6]:
%%writefile lab/.bazelversion
6.5.0


Overwriting lab/.bazelversion


In [7]:
%%writefile lab/.bazelrc
# Local build fixes for this environment (GCC 15 / Ubuntu 26.04, container as
# root -- no sandbox namespaces). See WORKSPACE / bazel/*.BUILD comments for
# why each dependency version was pinned or swapped.
build --spawn_strategy=local
build --copt=-std=gnu11
build --host_copt=-std=gnu11
# GCC 14+ promoted these from warnings to hard errors by default; the
# vendored C code (q3map2/picomodel, etc.) predates that and is otherwise
# harmless here.
build --copt=-Wno-error=incompatible-pointer-types
build --host_copt=-Wno-error=incompatible-pointer-types
build --copt=-Wno-error=implicit-function-declaration
build --host_copt=-Wno-error=implicit-function-declaration
build --copt=-Wno-error=int-conversion
build --host_copt=-Wno-error=int-conversion


Overwriting lab/.bazelrc


## 3. System build dependencies

The `apt-get` list from `lab/docs/users/build.md`, plus `build-essential` (a C/C++ compiler
toolchain isn't listed there but is required) and `bazelisk` itself (installed as `bazel`,
which resolves to `.bazelversion`'s pinned `6.5.0` automatically).


In [8]:
import subprocess

subprocess.run([
    "apt-get", "install", "-y",
    "libffi-dev", "gettext", "freeglut3-dev", "libsdl2-dev", "zip", "libosmesa6-dev",
    "python3-dev", "python3-numpy", "python3-pil", "build-essential",
], check=True)


Reading package lists...
Building dependency tree...


Reading state information...
libffi-dev is already the newest version (3.5.2-4).
gettext is already the newest version (0.23.2-1).
freeglut3-dev is already the newest version (3.4.0-6).
libsdl2-dev is already the newest version (2.32.10+dfsg-6).
zip is already the newest version (3.0-15ubuntu3).
libosmesa6-dev is already the newest version (25.1.7-1ubuntu3).
python3-dev is already the newest version (3.14.3-0ubuntu2).
python3-numpy is already the newest version (1:2.3.5+ds-3ubuntu1).
python3-pil is already the newest version (12.1.1-2ubuntu1.2).
build-essential is already the newest version (12.12ubuntu2.26.04.2).
Solving dependencies...


0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.


CompletedProcess(args=['apt-get', 'install', '-y', 'libffi-dev', 'gettext', 'freeglut3-dev', 'libsdl2-dev', 'zip', 'libosmesa6-dev', 'python3-dev', 'python3-numpy', 'python3-pil', 'build-essential'], returncode=0)

In [9]:
subprocess.run([
    "curl", "-fsSL", "-o", "/usr/local/bin/bazel",
    "https://github.com/bazelbuild/bazelisk/releases/latest/download/bazelisk-linux-amd64",
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/bazel"], check=True)


CompletedProcess(args=['chmod', '+x', '/usr/local/bin/bazel'], returncode=0)

## 4. Build the pip package

Must run with `env1` active on `PATH` *before* Bazel's server starts -- Bazel's `python_repo`
repository rule detects `python3` once (via `repository_ctx.execute(["python3", ...])`) and
caches that for the life of the server. If a server was already running under a different
Python (e.g. system Python from an earlier build), shut it down first so the next one picks
up `env1`'s `python3`.


In [10]:
import os

env = os.environ.copy()
subprocess.run(["bazel", "shutdown"], cwd="lab", env=env, check=False)
subprocess.run(
    ["bazel", "build", "-c", "opt", "--python_version=PY3",
     "//python/pip_package:build_pip_package"],
    cwd="lab", env=env, check=True,
)


Starting local Bazel server and connecting to it...


Loading: 


Loading: 
Loading: 0 packages loaded
Analyzing: target //python/pip_package:build_pip_package (1 packages loaded, 0 targets configured)


Analyzing: target //python/pip_package:build_pip_package (45 packages loaded, 1372 targets configured)


INFO: Analyzed target //python/pip_package:build_pip_package (86 packages loaded, 4485 targets configured).
INFO: Found 1 target...


...

INFO: From Linking libdmlab_headless_sw.so:
/usr/bin/x86_64-linux-gnu-ld.bfd: bazel-out/k8-opt/bin/external/lua_archive/liblua.pic.a(loslib.pic.o): in function `os_tmpname':
loslib.c:(.text.os_tmpname+0x24): warning: the use of `tmpnam' is dangerous, better use `mkstemp'
INFO: From Linking libdmlab_headless_hw.so:
/usr/bin/x86_64-linux-gnu-ld.bfd: bazel-out/k8-opt/bin/external/lua_archive/liblua.pic.a(loslib.pic.o): in function `os_tmpname':
loslib.c:(.text.os_tmpname+0x24): warning: the use of `tmpnam' is dangerous, better use `mkstemp'


/usr/bin/x86_64-linux-gnu-ld.bfd: bazel-out/k8-opt/bin/external/lua_archive/liblua.pic.a(loslib.pic.o): note: the message above does not take linker garbage collection into account
/usr/bin/x86_64-linux-gnu-ld.bfd: bazel-out/k8-opt/bin/external/lua_archive/liblua.pic.a(loslib.pic.o): note: the message above does not take linker garbage collection into account


[1,125 / 1,126] Executing genrule //:assets_pk3; 68s local


Target //python/pip_package:build_pip_package up-to-date:
  bazel-bin/python/pip_package/build_pip_package
INFO: Elapsed time: 76.947s, Critical Path: 71.02s
INFO: 1126 processes: 142 internal, 984 local.
INFO: Build completed successfully, 1126 total actions


CompletedProcess(args=['bazel', 'build', '-c', 'opt', '--python_version=PY3', '//python/pip_package:build_pip_package'], returncode=0)

## 5. Package the wheel and install it into this environment

In [11]:
subprocess.run(
    ["./bazel-bin/python/pip_package/build_pip_package", "/tmp/dmlab_pkg"],
    cwd="lab", check=True,
)


Wed Aug 12 10:53:01 KST 2026 : === Building wheel


/root/anaconda3/envs/env1/lib/python3.14/site-packages/setuptools/command/build_py.py:212: _Warning: Package 'deepmind_lab.baselab' is absent from the `packages` configuration.
!!

        ********************************************************************************
        ############################
        # Package would be ignored #
        ############################
        Python recognizes 'deepmind_lab.baselab' as an importable package[^1],
        but it is absent from setuptools' `packages` configuration.

        This leads to an ambiguous overall configuration. If you want to distribute this
        package, please make sure that 'deepmind_lab.baselab' is explicitly added
        to the `packages` configuration field.

        Alternatively, you can also rely on setuptools' discovery methods
        (for example by using `find_namespace_packages(...)`/`find_namespace:`
        instead of `find_packages(...)`/`find:`).

        You can read more about "package discove

/root/anaconda3/envs/env1/lib/python3.14/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Wed Aug 12 10:54:01 KST 2026 : === Output wheel file is in: /tmp/dmlab_pkg


CompletedProcess(args=['./bazel-bin/python/pip_package/build_pip_package', '/tmp/dmlab_pkg'], returncode=0)

In [12]:
import glob

wheel = glob.glob("/tmp/dmlab_pkg/*.whl")[0]
subprocess.run(["pip", "install", wheel, "--force-reinstall", "--no-deps"], check=True)
# Not pulled in automatically by the wheel's own dependency declarations in this setup.
subprocess.run(["pip", "install", "dm_env"], check=True)


Processing /tmp/dmlab_pkg/deepmind_lab-1.0-py3-none-any.whl


  Attempting uninstall: deepmind-lab
    Found existing installation: deepmind-lab 1.0
    Uninstalling deepmind-lab-1.0:
      Successfully uninstalled deepmind-lab-1.0


CompletedProcess(args=['pip', 'install', 'dm_env'], returncode=0)

## 6. Smoke test

Same check already run and confirmed working during development: `deepmind_lab` importable
alongside `torch`, an environment can be constructed and stepped, and an observation converts
to a CUDA tensor.


In [13]:
import numpy as np
import torch
import deepmind_lab

print("torch cuda:", torch.cuda.is_available())

env = deepmind_lab.Lab(
    "tests/empty_room_test", ["RGB_INTERLEAVED"], config={"width": "84", "height": "84"})
env.reset()
action = np.array([0, 0, 0, 1, 0, 0, 0], dtype=np.intc)
for _ in range(20):
    env.step(action, num_steps=4)
obs = env.observations()
print("obs shape:", obs["RGB_INTERLEAVED"].shape, obs["RGB_INTERLEAVED"].dtype)

t = torch.from_numpy(obs["RGB_INTERLEAVED"]).float().cuda()
print("as cuda tensor:", t.shape, t.device, t.mean().item())


torch cuda: True


obs shape: (84, 84, 3) uint8


as cuda tensor: torch.Size([84, 84, 3]) cuda:0 89.74187469482422
